# Hive 실습

- hive-server
- hive-metastore
- metastore-postgresql
- hive-cli

# 점검사항
- 하둡 네임노드 확인
    - <분산 데이터> 
        - 1424 NameNode - 마스터
        - 248 DataNode  - 슬레이브
    - <분산 처리>
        - 636 ResourceManager   - 마스터
        - 751 NodeManager       - 슬레이브 

In [93]:
!docker exec -it hadoop-master sh -c  'jps'

13040 Jps
1424 NameNode
248 DataNode
427 SecondaryNameNode
636 ResourceManager
751 NodeManager


* namenode  : 포맷진행 후 대몬 재실행

In [94]:
# hdfs namenode -format
# hdfs --daemon start namenode 
# jps

## 도커 실행 컨테이너 확인

In [95]:
!docker ps

CONTAINER ID   IMAGE                                     COMMAND                  CREATED      STATUS        PORTS                                                                    NAMES
8bd7d042bb44   bde2020/hive:2.3.2-postgresql-metastore   "entrypoint.sh tail …"   2 days ago   Up 2 days     10000/tcp, 10002/tcp                                                     hive-cli
92a2e25654d9   bde2020/hive:2.3.2-postgresql-metastore   "entrypoint.sh /bin/…"   2 days ago   Up 2 days     0.0.0.0:10000->10000/tcp, 10002/tcp                                      hive-server
3f5b81ab2e00   confluentinc/cp-kafka:7.6.0               "/etc/confluent/dock…"   2 days ago   Up 2 days     0.0.0.0:9092->9092/tcp, 0.0.0.0:19092->19092/tcp                         kafka
a01934f18c5b   bde2020/hive:2.3.2-postgresql-metastore   "entrypoint.sh /opt/…"   2 days ago   Up 2 days     10000/tcp, 0.0.0.0:9083->9083/tcp, 10002/tcp                             hive-metastore
74c8c53b5a04   t3-hadoop-hive-spark-spark-

## 도커 볼륨 확인

<pre> <font color=red><b>docker-compose.yml</b></font>
hive-metastore :   
    volumes:    
        - ./config/hadoop/core-site.xml:/etc/hadoop/core-site.xml
        - ./config/hadoop/hdfs-site.xml:/etc/hadoop/hdfs-site.xml
        - ./config/hive/hive-site.xml:/opt/hive/conf/hive-site.xml
        - ./config/hive/init-metastore.sh:/opt/hive/init-metastore.sh

hive-server:
    volumes:
      - ./config/hadoop/core-site.xml:/etc/hadoop/core-site.xml
      - ./config/hadoop/hdfs-site.xml:/etc/hadoop/hdfs-site.xml
      - ./config/hive/hive-site.xml:/opt/hive/conf/hive-site.xml

hive-cli:
    volumes:
        - ./config/hadoop:/config/hadoop
        - ./config/hive:/config/hive

## 하이브 네트워크 확인

<pre> <font color=red><b>docker-compose.yml</b></font>
networks:
  hive-net:
    name: hive-net
    driver: bridge
  hadoop:
    name: hadoop
    driver: bridge

In [96]:
!docker network ls

NETWORK ID     NAME                      DRIVER    SCOPE
d8e1c4a81953   bridge                    bridge    local
93a469843fb9   hadoop                    bridge    local
477c73993817   hive-net                  bridge    local
812073313c4a   host                      host      local
b69a932bb6ee   kafka-zookeeper_default   bridge    local
fd47c9ce2a82   none                      null      local
bc7cd291795b   t1-hadoop_hadoop-net      bridge    local


In [97]:
!docker inspect hive-metastore| grep "IPAddress"

            "SecondaryIPAddresses": null,
            "IPAddress": "",
                    "IPAddress": "172.20.0.11",
                    "IPAddress": "172.21.0.4",


In [98]:
!docker inspect hive-server| grep "IPAddress"

            "SecondaryIPAddresses": null,
            "IPAddress": "",
                    "IPAddress": "172.20.0.4",
                    "IPAddress": "172.21.0.5",


In [99]:
!docker inspect hive-cli| grep "IPAddress"

            "SecondaryIPAddresses": null,
            "IPAddress": "",
                    "IPAddress": "172.20.0.12",
                    "IPAddress": "172.21.0.6",


## 하이브 네트워크와 연결된 컨테이너 확인
- hive-server : 172.21.0.4/16
- hive-metastore : 172.21.0.3/16
- metastore-postgresql:172.21.0.2/16
- hive-cli : 172.21.0.5/16

In [100]:
! docker network inspect hive-net

[
    {
        "Name": "hive-net",
        "Id": "477c7399381753bb6799f567ebc31bc3fe2ff436a7f2247bac186f8d6010eda0",
        "Created": "2025-07-02T05:24:40.893472366Z",
        "Scope": "local",
        "Driver": "bridge",
        "EnableIPv4": true,
        "EnableIPv6": false,
        "IPAM": {
            "Driver": "default",
            "Options": null,
            "Config": [
                {
                    "Subnet": "172.21.0.0/16",
                    "Gateway": "172.21.0.1"
                }
            ]
        },
        "Internal": false,
        "Attachable": false,
        "Ingress": false,
        "ConfigFrom": {
            "Network": ""
        },
        "ConfigOnly": false,
        "Containers": {
            "8bd7d042bb4409df99f05f71ae3b24698ceb51ccff022b17f8295010701ba199": {
                "Name": "hive-cli",
                "EndpointID": "c0a7da74a80078bfd86ff5f316d35913b54cf7489e4bd4da9bf0c052502eac28",
                "MacAddress": "26:4b:98:07:ca:6a",

# **하이브 컨테이너 내부 진입**

## 하이브 버전 확인

In [101]:
!docker exec -it hive-server sh -c  'hive --version'

Hive 2.3.2
Git git://stakiar-MBP.local/Users/stakiar/Desktop/scratch-space/apache-hive -r 857a9fd8ad725a53bd95c1b2d6612f9b1155f44d
Compiled by stakiar on Thu Nov 9 09:11:39 PST 2017
From source with checksum dc38920061a4eb32c4d15ebd5429ac8a


## 하이브 홈디렉토리 확인

In [102]:
!docker exec -it hive-server sh -c 'cat $HIVE_HOME'

cat: /opt/hive: Is a directory


## 현재 작업 디렉토리 확인

In [103]:
!docker exec -it hive-server sh -c   'pwd'

/opt


In [104]:
!docker exec -it hive-server sh -c 'ls -l /opt/hive'

total 80
-rw-r--r-- 1 root staff 20798 Nov  9  2017 LICENSE
-rw-r--r-- 1 root staff   230 Nov  9  2017 NOTICE
-rw-r--r-- 1 root staff  1979 Nov  9  2017 RELEASE_NOTES.txt
drwxr-xr-x 3 root root   4096 Feb  5  2018 bin
drwxr-xr-x 2 root root   4096 Feb  5  2018 binary-package-licenses
drwxr-xr-x 1 root root   4096 Feb  5  2018 conf
drwxr-xr-x 4 root root   4096 Feb  5  2018 examples
drwxr-xr-x 7 root root   4096 Feb  5  2018 hcatalog
drwxr-xr-x 2 root root   4096 Feb  5  2018 jdbc
drwxr-xr-x 4 root root  20480 Feb  5  2018 lib
drwxr-xr-x 4 root root   4096 Feb  5  2018 scripts


## hive.metastore.uris
- <font color=red><b>C:\DC\t3-hadoop-hive-spark\config\hive\hive-site.xml

- hive.metastore.uris : thrift://hive-metastore:9083
- javax.jdo.option.ConnectionURL : jdbc:postgresql://metastore-postgresql:5432/metastore
- javax.jdo.option.ConnectionDriverName : org.postgresql.Driver
- javax.jdo.option.ConnectionUserName : hive
- javax.jdo.option.ConnectionPassword : hive

# [실습]

## Hive 접속 및 테이블 생성

In [105]:
!docker exec -i hive-server hive -e "CREATE DATABASE IF NOT EXISTS my_hive_db;  \
                                    USE my_hive_db;  \
                                    CREATE TABLE IF NOT EXISTS  emp_hive(id INT, name STRING); \
                                    SHOW TABLES;  \
                                    INSERT INTO emp_hive VALUES (1, 'Alice'), (2, 'Bob'); \
                                    INSERT INTO emp_hive VALUES (3, 'King'); \
                                    SELECT * FROM emp_hive; "

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]

Logging initialized using configuration in file:/opt/hive/conf/hive-log4j2.properties Async: true
OK
Time taken: 1.544 seconds
OK
Time taken: 0.016 seconds
OK
Time taken: 0.142 seconds
OK
emp_hive
emp_hive_ext_tbl
emp_parquet_tbl
emp_partition_tbl
emp_raw_hive_ext
emp_raw_tbl
emp_raw_tbl22
emp_raw_tbl33
iris_partition_tbl
lec_iris_tbl
Time taken: 0.105 seconds, Fetched: 10 row(s)
Query ID = root_20250704045940_70930617-a16b-4708-b435-fa0afe4325c5
Total jobs = 3
Launching Job 1 out of 3
Number of reduce tasks is set to 0 s

## 하둡내 HDFS에 Hive warehouse 디렉터리 확인 
- <font color=red><b>HDFS 내에서 **/user/hive/warehouses**는 hive 기본 저장 경로
- <font color=red><b>HDFS 내에서 **/user/hive/warehouses/하이브DB명/하이브table명/파일** 로 저장

In [106]:
! docker exec -it hadoop-master sh -c   'hdfs dfs -ls /user/hive/warehouse/my_hive_db.db/emp_hive/'

Found 10 items
-rwxr-xr-x   1 root supergroup         14 2025-07-03 00:48 /user/hive/warehouse/my_hive_db.db/emp_hive/000000_0
-rwxr-xr-x   1 root supergroup          7 2025-07-03 01:45 /user/hive/warehouse/my_hive_db.db/emp_hive/000000_0_copy_1
-rwxr-xr-x   1 root supergroup         14 2025-07-03 07:17 /user/hive/warehouse/my_hive_db.db/emp_hive/000000_0_copy_2
-rwxr-xr-x   1 root supergroup          7 2025-07-03 07:17 /user/hive/warehouse/my_hive_db.db/emp_hive/000000_0_copy_3
-rwxr-xr-x   1 root supergroup         14 2025-07-03 08:47 /user/hive/warehouse/my_hive_db.db/emp_hive/000000_0_copy_4
-rwxr-xr-x   1 root supergroup          7 2025-07-03 08:47 /user/hive/warehouse/my_hive_db.db/emp_hive/000000_0_copy_5
-rwxr-xr-x   1 root supergroup         14 2025-07-04 01:49 /user/hive/warehouse/my_hive_db.db/emp_hive/000000_0_copy_6
-rwxr-xr-x   1 root supergroup          7 2025-07-04 01:49 /user/hive/warehouse/my_hive_db.db/emp_hive/000000_0_copy_7
-rwxr-xr-x   1 root supergroup         1

## postgreSQL 메타스토어 조회 
- <font color=red><b>**** 직접적으로 metastore를 조회할 일은 없다.
- <font color=red><b>**** 데이터 자체는 Hive CLI, Beeline, 또는 Spark SQL 등에서 조회
- metastore : Hive 테이블, 파티션, 스키마 정보 등 메타데이터 관리
- 머신러닝 작업 시 : Spark, Hive, Presto 같은 도구를 통해 Hive 테이블에 접근
- JDBC, Spark SQL, HiveQL 쿼리를 이용해 데이터 R/W 시 메타데이터를 간접적으로 활용

<pre><b>
docker exec -it metastore-postgresql bash
psql -U hive -d metastore
\d "TBLS"
\d    
SELECT t.*, d.* FROM "TBLS" t JOIN "DBS" d ON t."DB_ID" = d."DB_ID" WHERE d."NAME" = 'my_hive_db';    

 TBL_ID | CREATE_TIME | DB_ID | LAST_ACCESS_TIME | OWNER | RETENTION | SD_ID | TBL_NAME |   TBL_TYPE    | VIEW_EXPANDED_TEXT | VIEW_ORIGINAL_TEXT | IS_REWRITE_ENABLED | DB_ID | DESC |                       DB_LOCATION_URI                       |    NAME    | OWNER_NAME | OWNER_TYPE
--------+-------------+-------+------------------+-------+-----------+-------+----------+---------------+--------------------+--------------------+--------------------+-------+------+-------------------------------------------------------------+------------+------------+------------
      1 |  1751134432 |     2 |                0 | root  |         0 |     1 | <font color=red>emp_hive</font> | MANAGED_TABLE |                    |                    | f                  |     2 |      | <font color=red>hdfs://hadoop-master:9000/user/hive/warehouse/my_hive_db.db</font> | <font color=red>my_hive_db</font> | root       | USER
(1 row)


## Hive 외부 데이터로 테이블 생성
- <font color=red><b>HDFS에 데이터를 올리고 Hive에서 외부 데이터를 사용해 테이블 생성

### 하둡에 csv 파일 준비

In [107]:
!docker exec -it hadoop-master sh -c "hdfs dfs -ls /tmp"   # HDFS 내 /tmp 디렉토리 조회

Found 1 items
drwx-wx-wx   - root supergroup          0 2025-07-02 05:40 /tmp/hive


In [108]:
!docker exec -it hadoop-master sh -c 'ls /tmp'  # 로컬(컨테이너 내부) /tmp 디렉토리 조회

hadoop-root
hadoop-root-datanode.pid
hadoop-root-namenode.pid
hadoop-root-nodemanager.pid
hadoop-root-resourcemanager.pid
hadoop-root-secondarynamenode.pid
hadoop-yarn-root
hsperfdata_root
jetty-0_0_0_0-8042-hadoop-yarn-common-3_3_6_jar-_-any-4366298280906260913
jetty-0_0_0_0-8088-hadoop-yarn-common-3_3_6_jar-_-any-1871388050590819860
jetty-0_0_0_0-9868-secondary-_-any-4167896385015674711
jetty-0_0_0_0-9870-hdfs-_-any-5362994076305980489
jetty-localhost-43269-datanode-_-any-4970534329969335924


In [109]:
!docker exec -it hadoop-master sh -c 'echo "1,John,5000\n2,Jane,6000\n3,Bob,5500" > emp_hive_ext.csv'
#CSV 파일 생성

In [110]:
!docker exec -it hadoop-master sh -c 'cat emp_hive_ext.csv'   #emp_hive_ext.csv 파일 내용 확인

1,John,5000
2,Jane,6000
3,Bob,5500


In [111]:
!docker exec -it hadoop-master sh -c 'ls /'

bin   emp_hive_ext.csv	home	  lib32   media  proc  sbin  tmp
boot  etc		iris.csv  lib64   mnt	 root  srv   usr
dev   hadoop		lib	  libx32  opt	 run   sys   var


### HDFS 저장 폴더 생성

In [112]:
!docker exec -it hadoop-master sh -c 'hdfs dfs -mkdir -p /my_hadoop_data'

In [113]:
!docker exec -it hadoop-master sh -c 'hdfs dfs -ls /'

Found 8 items
drwxr-xr-x   - root supergroup          0 2025-07-04 02:33 /lec
drwxr-xr-x   - root supergroup          0 2025-07-04 02:54 /my_hadoop_data
drwxr-xr-x   - root supergroup          0 2025-07-04 01:50 /my_hadoop_data.parquet
drwxr-xr-x   - root supergroup          0 2025-07-04 02:54 /my_hadoop_data22
drwxr-xr-x   - root supergroup          0 2025-07-04 01:50 /my_hadoop_data22.parquet
drwxr-xr-x   - root supergroup          0 2025-07-04 02:54 /my_hadoop_data_parquet22
drwx-wx-wx   - root supergroup          0 2025-07-02 05:40 /tmp
drwxr-xr-x   - root supergroup          0 2025-07-03 00:40 /user


### HDFS에 csv 파일 업로드
- http://localhost:9870
    : 좌측 "Utilities > Browse the file system" 사용해 업로드 가능

In [114]:
!docker exec -it hadoop-master sh -c 'hdfs dfs -put emp_hive_ext.csv /my_hadoop_data/'
#my_hadoop_data폴더에 csv저장

put: `/my_hadoop_data/emp_hive_ext.csv': File exists


In [115]:
!docker exec -it hadoop-master sh -c 'hdfs dfs -ls /my_hadoop_data/'

Found 2 items
-rw-r--r--   1 root supergroup         35 2025-07-03 02:12 /my_hadoop_data/emp_hive_ext.csv
drwxr-xr-x   - root supergroup          0 2025-07-04 02:54 /my_hadoop_data/emp_raw_hive


### Hive에서 외부 데이터를 사용해 파일 생성

* <font color=red><b>STORED AS TEXTFILE : csv → Hive 외부 테이블
* <font color=red><b>실습01_하이브_외부파일_테이블생성.txt 참조

* CREATE TABLE IF NOT EXISTS  emp_hive(id INT, name STRING);
* CREATE **EXTERNAL** TABLE IF NOT EXISTS  emp_hive(id INT, name STRING) ~ ~ **STORED AS TEXTFILE/STORED AS PARQUET** ~ ~ **LOCATION '/my_hadoop_data/**';

In [116]:
!docker exec -i hive-server hive -e "USE my_hive_db; \
CREATE EXTERNAL TABLE IF NOT EXISTS emp_hive_ext_tbl (empno INT, ename STRING, sal INT) \
ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' \
STORED AS TEXTFILE \
LOCATION '/my_hadoop_data/'; \
SELECT * FROM emp_hive_ext_tbl;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]

Logging initialized using configuration in file:/opt/hive/conf/hive-log4j2.properties Async: true
OK
Time taken: 1.027 seconds
OK
Time taken: 0.084 seconds
OK
Failed with exception java.io.IOException:java.io.IOException: Not a file: hdfs://hadoop-master:9000/my_hadoop_data/emp_raw_hive
Time taken: 0.994 seconds


* <font color=red><b>STORED AS PARQUET : CSV → Parquet 변환 후 → Hive 외부 테이블

In [117]:
#! pip install py4j

- [안된다] df = pd.read_csv("hdfs://hadoop-master:9000/my_hadoop_data/emp_hive_ext.csv")
- [된다] df = spark.read.csv("hdfs://hadoop-master:9000/my_hadoop_data/emp_hive_ext.csv", header=False, inferSchema=True)

In [118]:
# CSV → Parquet 변환
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("csv-to-parquet").getOrCreate()

psdf = spark.read.csv("hdfs://hadoop-master:9000/my_hadoop_data/emp_hive_ext.csv", header=False, inferSchema=True)
#df = spark.read.option("header", "false").csv("hdfs://hadoop-master:9000/my_hadoop_data/emp_hive_ext.csv")

psdf.write.mode("overwrite").parquet("hdfs://hadoop-master:9000/my_hadoop_data.parquet")

# 확인
df_parquet = spark.read.parquet("hdfs://hadoop-master:9000/my_hadoop_data.parquet")
df_parquet.show()

spark.stop()

25/07/04 14:00:10 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
+---+----+----+
|_c0| _c1| _c2|
+---+----+----+
|  1|John|5000|
|  2|Jane|6000|
|  3| Bob|5500|
+---+----+----+



또는 

In [119]:
! docker exec -it hadoop-master sh -c   'hdfs dfs -rm  -r -f  /my_hadoop_data.parquet'

Deleted /my_hadoop_data.parquet


In [120]:

#Spark를 활용해 HDFS에 있는 CSV 파일을 읽고 → 파케이(parquet) 포맷으로 저장하고 
# 다시 읽어서 처리하는 전체 흐름
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("new_session") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-master:9000") \
    .getOrCreate()


# from pyspark.sql.types import StructType, StructField, IntegerType, StringType
# schema = StructType([
#     StructField("id"    , IntegerType(), True),
#     StructField("name"  , StringType(), True),
#     StructField("salary", IntegerType(), True)
# ])
# df = spark.read.schema(schema).csv("/my_hadoop_data/emp_hive_ext.csv")
# df.show()

# 컬럼이름 지정
# HDFS에서 CSV 파일 읽기
psdf = spark.read.csv("hdfs://hadoop-master:9000/my_hadoop_data/emp_hive_ext.csv", header=False, inferSchema=True)
psdf = psdf.toDF("id", "name", "salary")

#psdf = spark.read.csv("/my_hadoop_data/emp_hive_ext.csv")
psdf.write.mode("overwrite").parquet("/my_hadoop_data.parquet")   #parquet으로 저장

# -------------------------------------------------------------------
df_parquet = spark.read.parquet("/my_hadoop_data.parquet")  #저장된 Parquet 파일 다시 읽기 
df_parquet = df_parquet.toDF("id22", "name22", "salary22")   #컬럼명 바꾸기
#.option("parquet.write_legacy_format", "true")
df_parquet.write.mode("overwrite").parquet("/my_hadoop_data22.parquet")
df_parquet.show()

spark.stop()

+----+------+--------+
|id22|name22|salary22|
+----+------+--------+
|   1|  John|    5000|
|   2|  Jane|    6000|
|   3|   Bob|    5500|
+----+------+--------+



* 하둡에서  /my_hadoop_data_parquet 확인

In [121]:
! docker exec -it hadoop-master sh -c   'hdfs dfs -ls /my_hadoop_data22.parquet'

Found 2 items
-rw-r--r--   1 root supergroup          0 2025-07-04 05:00 /my_hadoop_data22.parquet/_SUCCESS
-rw-r--r--   1 root supergroup        952 2025-07-04 05:00 /my_hadoop_data22.parquet/part-00000-03d4ace2-e1fa-457b-91f5-3578a0a64e5b-c000.snappy.parquet


*  Parquet 변환 후 → Hive 외부 테이블 생성

In [122]:
!docker exec -i hive-server hive -e "DROP TABLE IF EXISTS emp_parquet_tbl;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]

Logging initialized using configuration in file:/opt/hive/conf/hive-log4j2.properties Async: true
OK
Time taken: 1.03 seconds


In [123]:
!docker exec -i hive-server hive -e "USE my_hive_db; \
drop table emp_parquet_tbl; \
CREATE EXTERNAL TABLE IF NOT EXISTS emp_parquet_tbl (id22 INT, name22 STRING, salary22 INT) \
STORED AS PARQUET \
LOCATION 'hdfs://hadoop-master:9000/my_hadoop_data22.parquet/'; \
SELECT * FROM emp_parquet_tbl;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]

Logging initialized using configuration in file:/opt/hive/conf/hive-log4j2.properties Async: true
OK
Time taken: 0.963 seconds
OK
Time taken: 0.119 seconds
OK
Time taken: 0.112 seconds
OK
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
1	John	5000
2	Jane	6000
3	Bob	5500
Time taken: 1.121 seconds, Fetched: 3 row(s)


* <font color=red><b>PARTITIONED BY  : Hive 파티션 테이블 

In [124]:
!docker exec -it hadoop-master sh -c 'hdfs dfs -ls /lec'

Found 3 items
-rw-r--r--   1 root supergroup       5107 2025-07-03 03:19 /lec/iris.csv
drwxr-xr-x   - root supergroup          0 2025-07-04 02:33 /lec/iris.parquet
drwxr-xr-x   - root supergroup          0 2025-07-04 01:50 /lec/iris_partitioned


In [125]:
!docker exec -it hadoop-master sh -c 'hdfs dfs -cat /lec/iris.csv | head -n 5'

Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
1,5.1,3.5,1.4,0.2,Iris-setosa
2,4.9,3.0,1.4,0.2,Iris-setosa
3,4.7,3.2,1.3,0.2,Iris-setosa
4,4.6,3.1,1.5,0.2,Iris-setosa


In [126]:
from pyspark.sql import SparkSession

spark = ( SparkSession.builder 
    .appName("iris_partitioning") 
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-master:9000") 
    .getOrCreate() )

# 1. iris.csv 읽기
spdf = spark.read.csv("/lec/iris.csv", header=True, inferSchema=True)
spdf = spdf.toDF("seq", "sl", "sw", "pl","pw","Species")
spdf.show(3)

# 3. 파티션 기준으로 저장
spdf.write \
    .mode("overwrite") \
    .partitionBy("Species") \  #Species 기준으로 파티셔닝 저장
    .option("header", False) \
    .csv("/lec/iris_partitioned")

spark.stop()

+---+---+---+---+---+-----------+
|seq| sl| sw| pl| pw|    Species|
+---+---+---+---+---+-----------+
|  1|5.1|3.5|1.4|0.2|Iris-setosa|
|  2|4.9|3.0|1.4|0.2|Iris-setosa|
|  3|4.7|3.2|1.3|0.2|Iris-setosa|
+---+---+---+---+---+-----------+
only showing top 3 rows



In [127]:
!docker exec -it hadoop-master sh -c 'hdfs dfs -ls /lec'

Found 3 items
-rw-r--r--   1 root supergroup       5107 2025-07-03 03:19 /lec/iris.csv
drwxr-xr-x   - root supergroup          0 2025-07-04 02:33 /lec/iris.parquet
drwxr-xr-x   - root supergroup          0 2025-07-04 05:00 /lec/iris_partitioned


In [128]:
!docker exec -it hadoop-master sh -c 'hdfs dfs -ls /lec/iris_partitioned'

Found 4 items
drwxr-xr-x   - root supergroup          0 2025-07-04 05:00 /lec/iris_partitioned/Species=Iris-setosa
drwxr-xr-x   - root supergroup          0 2025-07-04 05:00 /lec/iris_partitioned/Species=Iris-versicolor
drwxr-xr-x   - root supergroup          0 2025-07-04 05:00 /lec/iris_partitioned/Species=Iris-virginica
-rw-r--r--   1 root supergroup          0 2025-07-04 05:00 /lec/iris_partitioned/_SUCCESS


In [129]:
!docker exec -i hive-server hive -e "USE my_hive_db; \
drop table iris_partition_tbl; \
CREATE EXTERNAL TABLE  IF NOT EXISTS iris_partition_tbl(seq INT, sl FLOAT, sw FLOAT,pl FLOAT, pw FLOAT) \
ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' \
STORED AS TEXTFILE \
LOCATION 'hdfs://hadoop-master:9000/lec/iris_partitioned/Species=Iris-setosa'; \
select * from iris_partition_tbl;"

#hdfs://hadoop-master:9000/lec/iris.csv

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]

Logging initialized using configuration in file:/opt/hive/conf/hive-log4j2.properties Async: true
OK
Time taken: 1.004 seconds
OK
Time taken: 0.102 seconds
OK
Time taken: 0.127 seconds
OK
1	5.1	3.5	1.4	0.2
2	4.9	3.0	1.4	0.2
3	4.7	3.2	1.3	0.2
4	4.6	3.1	1.5	0.2
5	5.0	3.6	1.4	0.2
6	5.4	3.9	1.7	0.4
7	4.6	3.4	1.4	0.3
8	5.0	3.4	1.5	0.2
9	4.4	2.9	1.4	0.2
10	4.9	3.1	1.5	0.1
11	5.4	3.7	1.5	0.2
12	4.8	3.4	1.6	0.2
13	4.8	3.0	1.4	0.1
14	4.3	3.0	1.1	0.1
15	5.8	4.0	1.2	0.2
16	5.7	4.4	1.5	0.4
17	5.4	3.9	1.3	0.4
18	5.1	3.5	1.4	0.3
19	5.7

# [실습] 

### Hive 접속 및 테이블 확인

In [130]:
!docker exec -i hive-server hive -e "USE my_hive_db;  \
                                    SHOW TABLES;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]

Logging initialized using configuration in file:/opt/hive/conf/hive-log4j2.properties Async: true
OK
Time taken: 1.802 seconds
OK
emp_hive
emp_hive_ext_tbl
emp_parquet_tbl
emp_partition_tbl
emp_raw_hive_ext
emp_raw_tbl
emp_raw_tbl22
emp_raw_tbl33
iris_partition_tbl
lec_iris_tbl
Time taken: 0.089 seconds, Fetched: 10 row(s)


## pandas를 활용한 parguet 만들기

In [131]:
# ! pip install pyarrow

In [132]:
import pandas as pd

In [133]:
df = pd.read_csv('iris.csv')
df.to_csv('iris_222.csv')
df.head(2)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa


In [134]:
df.to_parquet('iris_222.parquet', engine='pyarrow')
df22 = pd.read_parquet('iris_222.parquet')
df22.head(2)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa


## Spark + Hive 데이터 읽기

In [135]:
# PySpark에서 Hive와 연동한 뒤, 현재 Hive 데이터베이스에 있는 테이블 목록을 조회
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MySparkHiveExample") \
    .enableHiveSupport() \
    .getOrCreate()

psdf = spark.sql("use my_hive_db")
psdf = spark.sql("show tables")
psdf.show()

print(type(psdf))
spark.stop()

+----------+------------------+-----------+
| namespace|         tableName|isTemporary|
+----------+------------------+-----------+
|my_hive_db|          emp_hive|      false|
|my_hive_db|  emp_hive_ext_tbl|      false|
|my_hive_db|   emp_parquet_tbl|      false|
|my_hive_db| emp_partition_tbl|      false|
|my_hive_db|  emp_raw_hive_ext|      false|
|my_hive_db|       emp_raw_tbl|      false|
|my_hive_db|     emp_raw_tbl22|      false|
|my_hive_db|     emp_raw_tbl33|      false|
|my_hive_db|iris_partition_tbl|      false|
|my_hive_db|      lec_iris_tbl|      false|
+----------+------------------+-----------+

<class 'pyspark.sql.dataframe.DataFrame'>


## Spark + csv 데이터 읽기
<pre>
- pandas.DataFrame (단일)  vs pyspark.DataFrame (분산)
- pd.read_csv()               spark.read.csv()
- df.head()                   ps.show()
</pre>

In [136]:
# HDFS에 있는 CSV 파일을 Spark DataFrame으로 읽어와서, 상위 3줄
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MySparkParquetExample") \
    .getOrCreate()
     #.enableHiveSupport() \

# csv 읽기
##df =  pd.read_csv("hdfs://hadoop-master:9000/lec/iris.csv")
psdf = spark.read.option("header", True).csv("hdfs://hadoop-master:9000/lec/iris.csv")  #csv읽기
psdf.show(3)

+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 3 rows



## Spark를 활용한 parguet 만들기

In [137]:
# csv --> parquet 쓰기
psdf.write.mode("overwrite").parquet("hdfs://hadoop-master:9000/lec/iris.parquet")

# parquet 읽기
psdf22 = spark.read.parquet("hdfs://hadoop-master:9000/lec/iris.parquet")
psdf22.show(2)

spark.stop()

+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 2 rows



# [미션] Hadoop + Spark + Hive 통합

In [138]:
!docker exec -it hadoop-master sh -c 'ls -l /'

total 76
lrwxrwxrwx   1 root root    7 Apr  4 02:03 bin -> usr/bin
drwxr-xr-x   2 root root 4096 Apr 15  2020 boot
drwxr-xr-x   5 root root  360 Jul  2 05:24 dev
-rw-r--r--   1 root root   35 Jul  4 04:59 emp_hive_ext.csv
drwxr-xr-x   1 root root 4096 Jul  2 05:24 etc
drwxr-xr-x   2 root root 4096 Jun 25 11:51 hadoop
drwxr-xr-x   2 root root 4096 Apr 15  2020 home
-rwxrwxrwx   1 1001 1001 5107 May 24 19:21 iris.csv
lrwxrwxrwx   1 root root    7 Apr  4 02:03 lib -> usr/lib
lrwxrwxrwx   1 root root    9 Apr  4 02:03 lib32 -> usr/lib32
lrwxrwxrwx   1 root root    9 Apr  4 02:03 lib64 -> usr/lib64
lrwxrwxrwx   1 root root   10 Apr  4 02:03 libx32 -> usr/libx32
drwxr-xr-x   2 root root 4096 Apr  4 02:03 media
drwxr-xr-x   2 root root 4096 Apr  4 02:03 mnt
drwxr-xr-x   1 root root 4096 Jul  2 05:01 opt
dr-xr-xr-x 322 root root    0 Jul  2 05:24 proc
drwx------   1 root root 4096 Jul  2 05:26 root
drwxr-xr-x   1 root root 4096 Jul  2 05:25 run
lrwxrwxrwx   1 root root    8 Apr  4 02:03 sbin -

In [139]:
!docker exec -it hadoop-master sh -c 'hdfs dfs -ls /'

Found 8 items
drwxr-xr-x   - root supergroup          0 2025-07-04 05:00 /lec
drwxr-xr-x   - root supergroup          0 2025-07-04 02:54 /my_hadoop_data
drwxr-xr-x   - root supergroup          0 2025-07-04 05:00 /my_hadoop_data.parquet
drwxr-xr-x   - root supergroup          0 2025-07-04 02:54 /my_hadoop_data22
drwxr-xr-x   - root supergroup          0 2025-07-04 05:00 /my_hadoop_data22.parquet
drwxr-xr-x   - root supergroup          0 2025-07-04 02:54 /my_hadoop_data_parquet22
drwx-wx-wx   - root supergroup          0 2025-07-02 05:40 /tmp
drwxr-xr-x   - root supergroup          0 2025-07-03 00:40 /user


## Spark + HDFS 파일 읽기

In [140]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("my_mission") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-master:9000") \
    .getOrCreate()

spdf = spark.read.option("header", True).csv("/my_hadoop_data/emp_hive_ext.csv")
spdf.show(2)

+---+----+----+
|  1|John|5000|
+---+----+----+
|  2|Jane|6000|
|  3| Bob|5500|
+---+----+----+



## Spark + Hive 데이터 쓰기 (테이 블입력)

* <font color=red><b>TempView 생성  : spdf.createOrReplaceTempView("my_raw_data")
* <font color=red><b>TempView를 사용해 hive 테이블 생성
    - spark.sql(CREATE TABLE AAA     AS    select * from my_raw_data )

In [141]:
spdf.createOrReplaceTempView("my_raw_data")
spark.sql("USE my_hive_db;")
spark.sql("DROP TABLE IF EXISTS emp_raw_tbl")
spark.sql("CREATE TABLE  IF NOT EXISTS emp_raw_tbl AS       SELECT * FROM my_raw_data")
spark.sql("SELECT * FROM emp_raw_tbl").show()
   
# spark.stop()

25/07/04 14:00:59 WARN ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
+---+----+----+
|  1|John|5000|
+---+----+----+
|  2|Jane|6000|
|  3| Bob|5500|
+---+----+----+



* Hive 접속 및 테이블 확인

In [142]:
!docker exec -i hive-server hive -e "USE my_hive_db;  \
                                    SHOW TABLES;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]

Logging initialized using configuration in file:/opt/hive/conf/hive-log4j2.properties Async: true
OK
Time taken: 0.987 seconds
OK
emp_hive
emp_hive_ext_tbl
emp_parquet_tbl
emp_partition_tbl
emp_raw_hive_ext
emp_raw_tbl
emp_raw_tbl22
emp_raw_tbl33
iris_partition_tbl
lec_iris_tbl
Time taken: 0.092 seconds, Fetched: 10 row(s)


# [미션] Spark 결과 데이터 저장 방법
- HDFS
- 파일(csv, parquest)
- Hive내 테이블
- 외부 DB에 저장 (Oracle)

## 내부 데이터베이스(Hive) 저장

In [143]:
# HDFS : parquest 저장
spdf.write.mode("overwrite").parquet("hdfs://hadoop-master:9000/my_hadoop_data_parquet22")
print("----HDFS : parquest 저장 done ----")

#HDFS : csv 파일저장
spdf.write.mode("overwrite").option("header", True).csv("hdfs://hadoop-master:9000/my_hadoop_data22")
print("----HDFS : csv 파일저장 done ----")

# Hive 저장(1) (sql)
spdf.createOrReplaceTempView("my_raw_data22")
spark.sql("USE my_hive_db;")
spark.sql("CREATE TABLE  IF NOT EXISTS emp_raw_tbl22 AS       SELECT * FROM my_raw_data22")
print("----Hive 저장(1) (sql) done ----")

# Hive저장(2) Managed Table
spdf.write.mode("overwrite").saveAsTable("emp_raw_tbl33")
print("----Hive저장(2) Managed Table done ----")

# Hive저장(3) file path + extenal Table
spdf.write.mode("overwrite").option("path", "/my_hadoop_data/emp_raw_hive").saveAsTable("emp_raw_hive_ext")
print("----Hive저장(3) file path + extenal Table done ----")

----HDFS : parquest 저장 done ----
----HDFS : csv 파일저장 done ----
25/07/04 14:01:06 WARN ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
----Hive 저장(1) (sql) done ----
----Hive저장(2) Managed Table done ----
----Hive저장(3) file path + extenal Table done ----


* Hive 접속 및 테이블 확인

In [144]:
!docker exec -i hive-server hive -e "USE my_hive_db;  \
                                    SHOW TABLES;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]

Logging initialized using configuration in file:/opt/hive/conf/hive-log4j2.properties Async: true
OK
Time taken: 0.952 seconds
OK
emp_hive
emp_hive_ext_tbl
emp_parquet_tbl
emp_partition_tbl
emp_raw_hive_ext
emp_raw_tbl
emp_raw_tbl22
emp_raw_tbl33
iris_partition_tbl
lec_iris_tbl
Time taken: 0.088 seconds, Fetched: 10 row(s)


##  내부 데이터베이스(Redis)에 저장

### Redis: 빠른 속도가 필요한 곳에서 사용하는, 메모리 기반의 초고속 Key-Value 데이터베이스

### Redis
- 키초기화
    - redis-cli flushall
- 키조회
    - redis-cli SCAN 0 MATCH '*' COUNT 100
    - redis-cli KEYS '*'

In [145]:
psdf = spark.read.option("header", False).csv("hdfs://hadoop-master:9000/my_hadoop_data/emp_hive_ext.csv")
# 컬럼명 수동 지정
psdf = psdf.toDF("empno", "ename", "sal")
psdf.show()
psdf.printSchema()


#wsl버전
#docker exec -it hadoop-master hdfs dfs -ls /my_hadoop_data/emp_hive_ext.csv
#docker exec -it hadoop-master hdfs dfs -cat /my_hadoop_data/emp_hive_ext.csv

+-----+-----+----+
|empno|ename| sal|
+-----+-----+----+
|    1| John|5000|
|    2| Jane|6000|
|    3|  Bob|5500|
+-----+-----+----+

root
 |-- empno: string (nullable = true)
 |-- ename: string (nullable = true)
 |-- sal: string (nullable = true)



### 문자로 저장인 경우

In [146]:
#!docker ps
# redis:7.0-alpine  0.0.0.0:6379->6379/tcp                                                   redis

In [147]:
import redis

def save_to_redis_str(partition):
    r = redis.Redis(host='redis', port=6379)
    for row in partition:
        key = f"emp_str:{row['empno']}"
        value = f"{row['ename']},{row['sal']}"  # 쉼표로 구분된 문자열
        r.set(key, value)

psdf.foreachPartition(save_to_redis_str)
psdf.show()

[Stage 10:>                                                         (0 + 1) / 1]

+-----+-----+----+
|empno|ename| sal|
+-----+-----+----+
|    1| John|5000|
|    2| Jane|6000|
|    3|  Bob|5500|
+-----+-----+----+



* pyspark를 활용한 조회

In [148]:
# Redis 클라이언트 연결 (컨테이너 이름 기준)
r = redis.Redis(host='redis', port=6379)
# 단순 문자열 저장된 경우
print(r.get('emp_str:1')) 

# 디코드 : (byte → str 변환)
print(r.get('emp_str:1').decode())

b'John,5000'
John,5000


* CLI 사용한 조회

In [149]:
!docker exec -it redis redis-cli GET emp_str:1

"John,5000"


### 해시(Hash)로 저장된 경우

In [150]:
psdf.show()
psdf.printSchema()

+-----+-----+----+
|empno|ename| sal|
+-----+-----+----+
|    1| John|5000|
|    2| Jane|6000|
|    3|  Bob|5500|
+-----+-----+----+

root
 |-- empno: string (nullable = true)
 |-- ename: string (nullable = true)
 |-- sal: string (nullable = true)



In [151]:
def save_to_redis_hash(partition):
    r = redis.Redis(host='redis', port=6379)
    for row in partition:
        key = f"emp_hash:{row['empno']}"
        # value = f"{row['ename']},{row['sal']}"  # 쉼표로 구분된 문자열
        r.hset(key, mapping={
            "ename": row['ename'],
            "sal": row['sal']
        })

psdf.foreachPartition(save_to_redis_hash)


# Redis 클라이언트 연결 (컨테이너 이름 기준)
r = redis.Redis(host='redis', port=6379)

# 단순 문자열 저장된 경우
#print(r.get('emp:1'))  # b'John,5000' 같은 값이 나옴 (byte → str 변환 필요 시 decode())

# 해시(HASH)로 저장된 경우
print(r.hgetall('emp_hash:1'))  # {'ename': b'John', 'sal': b'5000'}


# 디코드 : (byte → str 변환
data = r.hgetall('emp_hash:1')
decoded = {k.decode(): v.decode() for k, v in data.items()}
print(decoded)

{b'ename': b'John', b'sal': b'5000'}
{'ename': 'John', 'sal': '5000'}


* CLI를 사용한 조회

In [152]:
! docker exec -it redis redis-cli HGETALL emp_hash:1

1) "ename"
2) "John"
3) "sal"
4) "5000"


### 키 조회 

In [153]:
! docker exec -it redis redis-cli SCAN 0 MATCH '*' COUNT 100

1) "0"
2) 1) "emp_str:1"
   2) "emp_str:2"
   3) "emp_hash:3"
   4) "emp_hash:2"
   5) "emp_str:3"
   6) "emp_hash:1"


또는

In [154]:
! docker exec -it redis redis-cli KEYS '*'

1) "emp_hash:3"
2) "emp_str:3"
3) "emp_hash:1"
4) "emp_str:1"
5) "emp_hash:2"
6) "emp_str:2"


### 키 삭제
- <font color=red><b>복구불가 : 사용시 주의주의

In [155]:
!docker exec -it redis redis-cli flushall  # DROP 복구불가

OK


In [156]:
! docker exec -it redis redis-cli KEYS '*'

(empty array)


## 외부 데이터베이스(Oracle)에 저장

### mysql

In [157]:
# df2.write \  
# .format("jdbc") \    
# .option("url", "jdbc:mysql://localhost:3306/mydb") \    
# .option("dbtable", "category_avg") \    
# .option("user", "root") \    
# .option("password", "1234") \    
# .save()

### Oracle

* 사전 세팅 (시스템 시간)

In [158]:
! cat /etc/timezone

Asia/Seoul


In [159]:
! ls -l /etc/localtime

lrwxrwxrwx 1 root root 30 Jul  4 12:38 /etc/localtime -> /usr/share/zoneinfo/Asia/Seoul


In [160]:
! ln -sf /usr/share/zoneinfo/Asia/Seoul /etc/localtime

In [161]:
! echo "Asia/Seoul" > /etc/timezone

* 사전 세팅(드라이버)
- 이미 volume으로 드라이버 배포되 있음(아래 명령 불필요)
- !wget https://repo1.maven.org/maven2/com/oracle/database/jdbc/ojdbc8/21.5.0.0/ojdbc8-21.5.0.0.jar -O ojdbc8.jar

In [162]:
! ls /opt/bitnami/spark/ext-jars

commons-pool2-2.11.1.jar
kafka-clients-3.3.1.jar
mysql-connector-j-8.0.33.jar
ojdbc8-19.3.0.0.jar
postgresql-42.3.8.jar
spark-sql-kafka-0-10_2.12-3.3.0.jar
spark-streaming-kafka-0-10-assembly_2.12-3.3.0.jar
spark-token-provider-kafka-0-10_2.12-3.3.0.jar


In [163]:
! pwd

/opt/bitnami/spark/workspace_pyspark


* <font color=red><b>spark-submit  : .py 실행 명령어

<pre> oracle_spark_test.py
from pyspark.sql import SparkSession

jars_path = "/opt/bitnami/spark/config/ext-jars/ojdbc8-19.3.0.0.jar"

spark = SparkSession.builder \
    .appName("my_oracle_test") \
    .config("spark.jars", jars_path) \
    .config("spark.driver.extraClassPath", jars_path) \
    .config("spark.executor.extraClassPath", jars_path) \
    .getOrCreate()

print("✅ spark.jars:", spark.sparkContext._conf.get("spark.jars"))
print("✅ listJars:", spark.sparkContext._jsc.sc().listJars())



터미널에서 실행 : spark-submit /opt/bitnami/spark/workspace_pyspark/spark_oracle_test.py

In [164]:
! ls

flask_web			 lec03_ml_basic.ipynb
iris.csv			 ojdbc8-19.3.0.0.jar
iris_222.csv			 spark_oracle_test.ipynb
iris_222.parquet		 spark_oracle_test.py
lec01.hadoop.ipynb		 svc01
lec02.hive_spark_practice.ipynb  실습01_하이브_외부파일_테이블생성.txt


In [165]:
! spark-submit --jars ./ojdbc8-19.3.0.0.jar spark_oracle_test.py

25/07/04 14:01:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/04 14:01:24 INFO SparkContext: Running Spark version 3.3.0
25/07/04 14:01:24 INFO ResourceUtils: ==============================================================
25/07/04 14:01:24 INFO ResourceUtils: No custom resources configured for spark.driver.
25/07/04 14:01:24 INFO ResourceUtils: ==============================================================
25/07/04 14:01:24 INFO SparkContext: Submitted application: my_oracle_test
25/07/04 14:01:24 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> name: cpus, amount: 1.0)
25/07/04 14:01:24 INFO ResourceProfile: Limiting resource is cpu
25/07/04 14:01:24 INFO ResourceProfileManage